In [11]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [6]:
df = pd.read_csv("gold_5m.csv")

df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)


df['day'] = df['date'].dt.day
df['hour'] = df['date'].dt.hour
df['minute'] = df['date'].dt.minute

df.drop(["date", "spread"],axis=1, inplace=True)

df.head()

,open,high,low,close,tick_volume,t_1,t_2,t_3,t_4,t_5,...,t_20,t_21,t_22,t_23,t_24,rsi,target,day,hour,minute
0,1520.09,1520.53,1519.95,1520.39,422,-0.48,-0.38,0.32,0.00,0.21,...,0.42,-0.03,0.02,0.21,-0.11,46.6,-0.54,2,9,10
1,1520.39,1520.52,1519.79,1519.85,273,0.30,-0.48,-0.38,0.32,0.00,...,-0.16,0.42,-0.03,0.02,0.21,41.1,-0.08,2,9,15
2,1519.85,1520.06,1519.57,1519.77,250,-0.54,0.30,-0.48,-0.38,0.32,...,-0.19,-0.16,0.42,-0.03,0.02,41.9,-0.20,2,9,20
3,1519.77,1520.02,1519.18,1519.57,221,-0.08,-0.54,0.30,-0.48,-0.38,...,-0.10,-0.19,-0.16,0.42,-0.03,41.5,0.19,2,9,25
4,1519.49,1520.31,1519.06,1519.68,432,-0.20,-0.08,-0.54,0.30,-0.48,...,0.34,-0.10,-0.19,-0.16,0.42,42.5,0.43,2,9,30


In [7]:
X = df.drop(["target"], axis=1).values
y = df["target"].values.reshape(-1, 1)

In [8]:
X_scaled = MinMaxScaler().fit_transform(X)
y_scaled = MinMaxScaler().fit_transform(y)

In [9]:
def sequences(X, y, window_size=30):
    Xs, ys = [], []
    for i in range(len(X) - window_size):
        Xs.append(X[i:i+window_size])
        ys.append(y[i+window_size])
    return np.array(Xs), np.array(ys)

In [10]:
X_seq, y_seq = sequences(X_scaled, y_scaled)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, random_state=42)

In [15]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

trainDS = TimeSeriesDataset(X_train, y_train)
testDS = TimeSeriesDataset(X_test, y_test)

trainLoad = DataLoader(trainDS, batch_size=64, shuffle=True)
testLoad = DataLoader(testDS, batch_size=64, shuffle=False)

In [20]:
class LSTMModel(nn.Module):
    def __int__(self, input_size=33, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out


device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

model = LSTMModel().to(device)

In [ ]:
lossfunction = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float("inf")
patience, patience_counter = 10, 0
numepoch = 50

for epoch in range(numepoch):
    model.train()
    train_loss = 0
    for X_batch, y_batch in trainLoad:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(X_batch)

        loss = lossfunction(y_pred, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(trainDS)


    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in testLoad:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_pred = model(X_batch)
            loss = lossfunction(y_pred, y_batch)
            val_loss += loss.item() * X_batch.size(0)

    val_loss /= len(testDS)
    print(f"Epoch {epoch+1}/{numepoch} - train_loss: {train_loss:.6f} - val_loss: {val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_lstm.pt')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping.")
            break

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

model.load_state_dict(torch.load('best_lstm.pt'))
model.eval()

preds, actuals = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_pred = model(X_batch).cpu().numpy()
        preds.append(y_pred)
        actuals.append(y_batch.numpy())

preds = np.concatenate(preds)
actuals = np.concatenate(actuals)

# Inverse scaling biar hasil dalam skala harga asli
preds_inv = scaler_y.inverse_transform(preds)
actuals_inv = scaler_y.inverse_transform(actuals)

print("R2  :", r2_score(actuals_inv, preds_inv))
print("RMSE:", mean_squared_error(actuals_inv, preds_inv, squared=False))